# Klasifikasi DemogPairs Menggunakan ViT (Emosi) & SVM

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
features = joblib.load('features/demogpairs_vit-emotion.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
from sklearn.decomposition import PCA

grid_params = [
    {
        'scaler': [None, MinMaxScaler()],  # EN: scaling option / ID: opsi scaling
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],  # EN: with and without PCA / ID: dengan dan tanpa PCA
        
        'classifier': [SVC()],  # EN: SVM model / ID: model SVM
        
        'classifier__C': [0.01, 0.1, 1, 10],  # EN: regularization / ID: regularisasi
        'classifier__kernel': ['rbf', 'poly', 'linear'],  # EN: kernel types / ID: jenis kernel
        
        'classifier__gamma': ['scale', 'auto'],  # EN: gamma / ID: gamma
        'classifier__degree': [2, 3],  # EN: degree / ID: derajat
        
        'classifier__tol': [1e-3],  # EN: tolerance / ID: toleransi
        'classifier__probability': [True],  # EN: probability / ID: probabilitas
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),  # EN: optional scaler / ID: scaler opsional
    ('pca', None),     # EN: optional PCA / ID: PCA opsional
    ('classifier', None)  # EN: classifier / ID: classifier
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro'
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

SVC: 288 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models,
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix='models/clf_demogpairs_svm_vit-emotion_',
    results_path='results/demogpairs_svm_vit-emotion_'
)

sorted_results = pd.DataFrame(evaluation_results).sort_values(by='test_accuracy', ascending=False).to_dict('records')
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: SVC


{'classifier': 'SVC', 'classifier__C': 10, 'classifier__degree': 2, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf', 'classifier__probability': True, 'classifier__tol': 0.001, 'pca': None, 'scaler': None}


Accuracy  : 0.9018518518518519
Precision : 0.9019744490108398
Recall    : 0.9018518518518519
F1 Score  : 0.9017100978866387
               precision    recall  f1-score   support

Asian_Females     0.9290    0.9444    0.9366       360
  Asian_Males     0.8952    0.9250    0.9098       360
Black_Females     0.8733    0.9000    0.8865       360
  Black_Males     0.9244    0.9167    0.9205       360
White_Females     0.9094    0.8639    0.8860       360
  White_Males     0.8807    0.8611    0.8708       360

     accuracy                         0.9019      2160
    macro avg     0.9020    0.9019    0.9017      2160
 weighted avg     0.9020    0.9019    0.9017      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9787037037037037,0.9289617486338798,0.9444444444444444,0.9366391184573003,360
Asian_Males,0.9694444444444444,0.8951612903225806,0.925,0.9098360655737704,360
Black_Females,0.961574074074074,0.8733153638814016,0.9,0.8864569083447332,360
Black_Males,0.9736111111111111,0.9243697478991597,0.9166666666666666,0.9205020920502092,360
White_Females,0.9629629629629629,0.9093567251461988,0.8638888888888889,0.886039886039886,360
White_Males,0.9574074074074074,0.8806818181818182,0.8611111111111112,0.8707865168539326,360


Confusion matrix saved: images\cm_svm_vit-emotion_SVC.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               340                 1                 9                 4                 6                 0
         Asian_Males                 1               333                 0                 5                10                11
       Black_Females                 4                 1               324                15                 2                14
         Black_Males                 8                 5                16               330                 1                 0
       White_Females                11                15                 4                 2               311                17
         White_Males                 2                17                18                 1                12               310


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
SVC,models/clf_demogpairs_svm_vit-emotion_SVC.pkl,"{'classifier': 'SVC', 'classifier__C': 10, 'classifier__degree': 2, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf', 'classifier__probability': True, 'classifier__tol': 0.001, 'pca': None, 'scaler': None}",0.9018518518518519,0.9017100978866387,0.9019744490108398,0.9018518518518519,288


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_svm_vit-emotion_SVC.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 14209.0,
 'days': 0,
 'hours': 3,
 'minutes': 56,
 'seconds': 49.0,
 'text': '0 hari 3 jam 56 menit 49.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 116962.0,
 'days': 1,
 'hours': 8,
 'minutes': 29,
 'seconds': 22.0,
 'text': '1 hari 8 jam 29 menit 22.0 detik'}

In [9]:
def remove_keys(data, keys_to_remove, inplace=False):
    # EN: if not inplace, create a copy / ID: jika tidak inplace, buat salinan
    if not inplace:
        data = data.copy()
    
    # EN: loop through keys and remove safely / ID: loop key dan hapus dengan aman
    for key in keys_to_remove:
        data.pop(key, None)  # EN: avoid error if key not found / ID: aman jika key tidak ada
    
    return data

def dict_to_sentence(d):
    # EN: convert key-value pairs into readable parts / ID: ubah key-value jadi bagian kalimat
    parts = [f"{k}={v}" for k, v in d.items()]
    
    # EN: join all parts into one sentence / ID: gabungkan jadi satu kalimat
    sentence = ", ".join(parts)
    
    return sentence

fold_displays = []
for r in [remove_keys(r, ['No', 'F1 Score Mean', 'Precision Mean', 'Recall Mean', 'Train Time Mean']) for r in fold_results]:
    r['Params'] = dict_to_sentence(remove_keys(r['Params'], ['classifier'])).replace('classifier__', '')
    r['Mean'] = r['Accuracy Mean']
    del r['Accuracy Mean']
    fold_displays.append(r)
fold_displays = [{'No': idx + 1, **r} for idx, r in enumerate(sorted(fold_displays, key=lambda x: x['Mean'], reverse=True))]
_dtable = u.display_table(fold_displays, n_items=[3, 3, 3, 3], column_widths=['5%', '65%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean
1,"C=10, degree=3, gamma=scale, kernel=rbf, probability=True, tol=0.001, pca=None, scaler=None",0.8883,0.8895,0.8762,0.8819,0.8831,0.8838
2,"C=10, degree=2, gamma=scale, kernel=rbf, probability=True, tol=0.001, pca=None, scaler=None",0.8883,0.8895,0.8762,0.8819,0.8831,0.8838
3,"C=10, degree=2, gamma=scale, kernel=rbf, probability=True, tol=0.001, pca=None, scaler=MinMaxScaler",0.8889,0.886,0.8796,0.8773,0.886,0.8836
...,...,...,...,...,...,...,...
96,"C=0.01, degree=2, gamma=scale, kernel=linear, probability=True, tol=0.001, pca=PCA, scaler=None",0.8449,0.8461,0.8432,0.8391,0.8565,0.8459
97,"C=0.01, degree=3, gamma=auto, kernel=linear, probability=True, tol=0.001, pca=PCA, scaler=None",0.8449,0.8461,0.8432,0.8391,0.8565,0.8459
98,"C=0.01, degree=3, gamma=scale, kernel=linear, probability=True, tol=0.001, pca=PCA, scaler=None",0.8449,0.8461,0.8432,0.8391,0.8565,0.8459
...,...,...,...,...,...,...,...
191,"C=0.1, degree=2, gamma=auto, kernel=rbf, probability=True, tol=0.001, pca=PCA, scaler=None",0.7951,0.8003,0.7766,0.7911,0.8032,0.7933
192,"C=1, degree=2, gamma=auto, kernel=rbf, probability=True, tol=0.001, pca=None, scaler=MinMaxScaler",0.7992,0.7986,0.7737,0.7894,0.805,0.7932
